Before running the notebook, add dataset: 
Go to File -> Add data.
Search for https://www.kaggle.com/datasets/grassknoted/asl-alphabet . 
Attach this file, the code will read from it using kaggle input 


In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from torchvision import models
import torch.nn as nn
import torch
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 


data_dir = '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train'
#  We used ResNet-50 model for ASL

#  ResNet-50 normalize mean and std values
#  mean
#  Standard Deviation
mean_val = [0.485, 0.456, 0.406]
std_val  = [0.229, 0.224, 0.225]


In [ ]:
# Step 1
# We use data augmentation to avoid overfitting and to train
# the model to be more effective with diverse, real-world inputs
# Only for training transform, we do need augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean_val, std_val)
])
# Load validation ASL dataset
## Used to tune hyperparameters (learning rate, batch size, number of epochs...)
## Monitor overfitting during training

# data partiion spilt into train, validation and test
dataset = datasets.ImageFolder(data_dir, transform = transform)
total_size = len(dataset)
val_size = int(0.15 * total_size)
train_size = int(0.7 * total_size)
test_size = total_size - train_size - val_size

classes= dataset.classes
print(f'Total: {total_size}, len of training images: {train_size}, validation images :{val_size} and test images : {test_size}' )

# Load training ASL dataset
## This is the dataset the model learns from.
## The model adjusts its weights and biases using this data.

# split dataset randomly but reproducable
training_dataset, validation_dataset, test_dataset  = random_split(
    dataset, [train_size, val_size, test_size], generator = torch.Generator().manual_seed(42))

training_dataset_loader = DataLoader(training_dataset, batch_size = 32, shuffle = True)
validation_dataset_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False)
test_dataset_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

# Load test ASL dataset
## Measures final performance of the trained model.
## Simulates how the model will perform in the real world.




In [ ]:
# Step 2
# Load Pretrained ResNet-50 model
# num_classes = 26 letters + 3 special signs
num_classes = 29  
resnet50_model = models.resnet50(weights = models.ResNet50_Weights.IMAGENET1K_V2)

# Step 3
# ResNet-50 is trained on ImageNet, which has 1000 categories
# Its last layer - fully connected (fc) produces a vector of length 1000
# For ASL, we only need 29 classes, So we replace the original FC layer with a new one that outputs 29 values
resnet50_model.fc = nn.Linear(resnet50_model.fc.in_features, num_classes)

## Checks if a GPU (CUDA) is available.
## If yes, training/inference will run on GPU (much faster).
## Otherwise, it falls back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Transfers the model to the selected device (GPU or CPU).
resnet50_model = resnet50_model.to(DEVICE)

# This is a loss function.
## It measures the difference between the model’s predicted class probabilities and the true labels.
## During training, the model learns by minimizing the loss.
## The optimizer (e.g., Adam, SGD) uses this loss to adjust weights via backpropagation.
## Without a loss function, the model has no signal to learn — it won’t know how “wrong” its predictions are.
## loss_fun_object(outputs, labels) calculate the actual loss (scalar tensor)
loss_fun_object = nn.CrossEntropyLoss()


In [ ]:
# train function
def model_training(resnet50_model_loc, dataset_loader, optimizer, loss_fun_object):
    # enables Drop out and Batch Normalization
    resnet50_model_loc.train()

    total_loss = 0
    correct = 0
    total = 0    

    # loop through batches of data
    for images, labels in dataset_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)        

        optimizer.zero_grad() # clear previous gradient
        outputs = resnet50_model_loc(images)
        loss = loss_fun_object(outputs, labels)
        loss.backward() # calculate loss
        optimizer.step() # update the weights

        # metrics for the epoch
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataset_loader), correct / total


In [ ]:

# validation function
## This line calculates the validation accuracy, 
## which tells how well the model performs on unseen data during training.

## Validation accuracy tells how well the model performs on unseen data and prevents overfitting.
def evaluate(resnet50_model_loc, dataset_loader):

    # disable dropout and batch normalization
    resnet50_model_loc.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    # disbale gradient calculation
    with torch.no_grad():
        for images, labels in dataset_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE) 
            
            outputs = resnet50_model_loc(images) # only forward pass 
            loss = loss_fun_object(outputs, labels)
            total_loss += loss.item() # calculate loss

            # calculate accuarcy 
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss/ len(dataset_loader), correct / total

In [ ]:

# Step 4

# list for tracking accuracy
train_losses = []
train_accs = []
val_losses = []
val_accs = []
# Phase 1 → Train only the last layer (fast, stable, low risk of overfitting)
# Phase 2 → Train last few layers + FC layer (learns ASL-specific features)

# Phase 1 — Feature Extraction and freeze all base layers
# Loops through all parameters in ResNet-50

# Loops through all parameters and freeze all layers
for param in resnet50_model.parameters():
    param.requires_grad = False

# Only train the last FC layer
## for param in resnet50_model.fc.parameters():
## param.requires_grad = True
resnet50_model.fc.weight.requires_grad = True
resnet50_model.fc.bias.requires_grad = True

## This line creates an Adam optimizer that will update only the weights of the 
## last fully connected (FC) layer of the ResNet-50 model.

## Adam is an optimization algorithm used during training.
## It adjusts the model’s weights to reduce the loss.
## This tells the optimizer: “Train only the FC layer.”

## This line creates an Adam optimizer that updates only the last FC layer of ResNet-50, 
## using a learning rate of 0.001.
# learning rate == 0.001
optimizer = optim.Adam(resnet50_model.fc.parameters(), 0.001)

## This creates a learning-rate scheduler that automatically reduces the learning rate when the model stops improving.
## scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2)

## 1 epoch = the model sees all 30,000 images once
## 5 epochs = the model sees the dataset 5 times

## Why do we train for multiple epochs?
## Because the model cannot learn everything by seeing the data only once.

## Each epoch helps the model:
## - reduce the loss
## - adjust weights
## - improve accuracy

## Train only the last FC layer for 4 epochs while all other layers are frozen.

phase_one_epochs = 4

for epoch in range(phase_one_epochs):
    # train and validate
    training_loss, training_accuracy = model_training(resnet50_model, 
                                            training_dataset_loader, 
                                            optimizer, 
                                            loss_fun_object)
    validation_loss, validation_accuracy = evaluate(resnet50_model, validation_dataset_loader)

    if validation_accuracy < 0.05 and epoch > 1:
        print("Stopping early — validation accuracy too low.")
        break


    print(f"Epoch {epoch+1}/4 | "
          f"Loss: {training_loss:.4f} | Train Acc: {training_accuracy:.4f} | Val Loss: {validation_loss:.4f} | Val Acc: {validation_accuracy:.4f}")




In [ ]:
# Phase 2 — Fine-Tuning and unfreeze last few layers
## ResNet-50, the convolutional part is divided into 4 main blocks
# Unfreeze last ResNet block
# layer4 = the last block of ResNet-50 (top-level features, closest to output)
## Early layers (layer1–layer3) detect generic features: edges, textures
## layer4 detects high-level, task-specific features
## Unfreezing only layer4 helps the model adapt to ASL without overfitting or long training time

for name, param in resnet50_model.named_parameters():
    if "layer4" in name:
        param.requires_grad = True


# Keep FC layer trainable
resnet50_model.fc.weight.requires_grad = True
resnet50_model.fc.bias.requires_grad = True

# Optimizer for fine-tuning
## a small function that keeps only parameters that are trainable
## selects only those parameters where requires_grad = True
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, resnet50_model.parameters()), 
    0.0001
)
phase_two_epochs = 5


for epoch in range(phase_two_epochs):
    # train and validate
    training_loss, training_accuracy = model_training(resnet50_model, 
                                           training_dataset_loader, 
                                           optimizer, 
                                           loss_fun_object)
    validation_loss, validation_accuracy = evaluate(resnet50_model, validation_dataset_loader)

    # save history
    train_losses.append(training_loss )
    train_accs.append(training_accuracy * 100)
    val_losses.append(validation_loss )
    val_accs.append(validation_accuracy * 100)

    print(f"Epoch {epoch+1}/5 | "
          f"Loss: {training_loss:.4f} | Train Acc: {training_accuracy:.4f} | Val Loss: {validation_loss:.4f} | Val Acc: {validation_accuracy:.4f}")

     
    if validation_accuracy < 0.05 and epoch > 1:
        print("Stopping early — validation accuracy too low.")
        break

In [ ]:
# plot training and validation graph over epochs
epochs = range(1, phase_two_epochs + 1)

plt.figure(figsize=(12, 5))
plt.suptitle('Transfer Learning Training and Validation Metrics', fontsize = 16)
plt.subplot(1, 2, 1)
plt.plot(epochs, train_accs, label="Train Acc")
plt.plot(epochs, val_accs, label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.xlabel("Epochs")

plt.subplot(1, 2, 2)
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Val Loss")
plt.legend()
plt.title("Loss")
plt.xlabel("Epochs")


plt.show()

In [ ]:
# calculates the overall metrics 
def final_evaluation(resnet50_model_loc, dataset_loader, DEVICE, classes):

    # disables dropout and batch normalisation
    resnet50_model_loc.eval()

    all_preds = []
    all_labels = []

    # disables gradient calculation
    with torch.no_grad():

        # loop through test data in batches
        for images, labels in dataset_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)   
            
            outputs = resnet50_model_loc(images) # get raw outputs
            _, preds = torch.max(outputs, 1) # get predictions
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    # calculates metrics 
    accuracy = accuracy_score(all_labels, all_preds)
    per_accuracy = accuracy * 100
    f1 = f1_score(all_labels, all_preds, average='macro' )
    cm = confusion_matrix(all_labels, all_preds )

    print('Final metircs on Test Dataset')
    print(f'Test accuracy: {per_accuracy:.4f}')
    print(f'F1 Score: {f1:.4f}')

    print('Classification Report:')
    print(classification_report(all_labels,all_preds , target_names= classes))

    plt.figure(figsize=(16,16))
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted", fontsize = 12)
    plt.ylabel("True", fontsize = 12)
    plt.title("Transfer Learning ASL Alphabet Confusion Matrix", fontsize = 16)


    return per_accuracy, cm, f1

In [ ]:
print("\n Final evaluation:")
# Step 5 — Accuracy calculation
## Accuracy tells us how well the model predicts ASL letters on unseen data.
test_acc, cm, test_f1 = final_evaluation(resnet50_model, test_dataset_loader, DEVICE, classes )



In [ ]:
# calculate per class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for cls, acc in zip(classes, per_class_acc):
    print(f"{cls:10s}: {acc*100:.2f}%")

In [ ]:
# Step 6 — Model saving
torch.save(resnet50_model.state_dict(), "asl_resnet50.pth")
print("Transfer Learning Model saved!")


In [ ]:
# plot the prediction on the test batch

# Take one batch and predict
vis_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)
dataiter = iter(vis_loader)
images, labels = next(dataiter)

# Move to GPU
images = images.to(DEVICE)
labels = labels.to(DEVICE)

# Predict
resnet50_model.eval()
with torch.no_grad():
    outputs = resnet50_model(images)
    _, preds = outputs.max(1)

def imshow(img, title):
    # Undo the ResNet normalization 
    # Mean: [0.485, 0.456, 0.406], Std: [0.229, 0.224, 0.225]
    img = img.cpu().numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1) # Clip to valid range
    
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

# Plot 6 images
plt.figure(figsize=(12, 6))
plt.suptitle('Transfer Visual Sample Predictions on Test Data')
for i in range(6):
    plt.subplot(2, 3, i+1)
    imshow(images[i], title=f"Pred: {classes[preds[i]]}\nTrue: {classes[labels[i]]}")
plt.tight_layout()

plt.show()